In [1]:
import pandas as pd
import numpy as ny
import pyodbc as pys
import re
from datetime import datetime
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [2]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
             
             select NUM_IDENTIFICACION,CANT_MATERIAS, CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS, CICLO, COD_UNIDAD,
                CASE WHEN COD_PENSUM LIKE '%D' THEN REPLACE (COD_PENSUM ,'D','')
                WHEN COD_PENSUM LIKE '%N' THEN REPLACE (COD_PENSUM ,'N','')
                ELSE COD_PENSUM END AS PENSUM_L 
      
            from openquery([172.16.1.175],
            'select * from iceberg.CICLOS_X_ESTUDIANTE')
                      
        
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

### Reconocimiento de titulo

In [3]:
df[df['NUM_IDENTIFICACION']=='1018477214']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L
484314,1018477214,23.0,23.0,23.0,TECNOLOGO,VP15G,VG002
484315,1018477214,20.0,20.0,0.0,PROFESIONAL,VP15G,VG002
484316,1018477214,29.0,29.0,29.0,TECNICO,VP15G,VG002


### Reconocimiento de tecnico

In [4]:
df[df['NUM_IDENTIFICACION']=='1234196804']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L
1566575,1234196804,15.0,7.0,0.0,TECNOLOGO,VGM62,3040
1566576,1234196804,23.0,0.0,NaN,PROFESIONAL,VGM62,3040
1566577,1234196804,29.0,29.0,29.0,TECNICO,VGM62,3040


### Homologante

In [5]:
df[df['NUM_IDENTIFICACION']=='1036935311']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L
1089569,1036935311,0.0,0.0,NaN,PROFESIONAL,CE267,CE267
1089570,1036935311,0.0,0.0,NaN,TECNICO,CE267,CE267
1089571,1036935311,0.0,0.0,NaN,TECNOLOGO,CE267,CE267
1089572,1036935311,23.0,23.0,23.0,TECNOLOGO,VP15G,VG002
1089573,1036935311,20.0,20.0,1.0,PROFESIONAL,VP15G,VG002
1089574,1036935311,29.0,29.0,29.0,TECNICO,VP15G,VG002


#### Normal

In [6]:
df[df['NUM_IDENTIFICACION']=='50876215']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L
2214856,50876215,18.0,5.0,0.0,TECNOLOGO,VTM14,6017
2214857,50876215,29.0,0.0,NaN,PROFESIONAL,VTM14,6017
2214858,50876215,25.0,24.0,0.0,TECNICO,VTM14,6017


In [7]:
def asignar_marca(row):
    if pd.isna(row['CANT_MAT_HOMOLOGADAS']) or row['CANT_MAT_HOMOLOGADAS'] == 0:
        return 'Normal'
    elif row['CANT_MATERIAS'] == row['CANT_MAT_HOMOLOGADAS']:
        return 'Reconocimiento'
    else:
        return 'Homologante'

df['Marca'] = df.apply(asignar_marca, axis=1)

In [8]:
len(df[df['NUM_IDENTIFICACION']=='83183336'])

21

In [9]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT DISTINCT COD_PROG_TECNICO,COD_PROG_TECNOLOGO,COD_PROG_PROFESIONAL,PENSUM 
		    FROM CUN.MALLASLAURA 
      
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_mallas_l = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [10]:
df_a = pd.merge(df, df_mallas_l, how='left', 
                left_on=['COD_UNIDAD', 'PENSUM_L'], 
                right_on=['COD_PROG_TECNICO', 'PENSUM'])

df_a = pd.merge(df_a, df_mallas_l, how='left', 
                left_on=['COD_UNIDAD', 'PENSUM_L'], 
                right_on=['COD_PROG_TECNOLOGO', 'PENSUM'])

df_a = pd.merge(df_a, df_mallas_l, how='left', 
                left_on=['COD_UNIDAD', 'PENSUM_L'], 
                right_on=['COD_PROG_PROFESIONAL', 'PENSUM'])

In [11]:
import numpy as np

def obtener_codigo(row):
    if row['CICLO'] == 'TECNICO':
        return next((x for x in [row['COD_PROG_TECNICO_x'], row['COD_PROG_TECNICO_y'], row['COD_PROG_TECNICO']] if pd.notna(x)), np.nan)
    elif row['CICLO'] == 'TECNOLOGO':
        return next((x for x in [row['COD_PROG_TECNOLOGO_x'], row['COD_PROG_TECNOLOGO_y'], row['COD_PROG_TECNOLOGO']] if pd.notna(x)), np.nan)
    elif row['CICLO'] == 'PROFESIONAL':
        return next((x for x in [row['COD_PROG_PROFESIONAL_x'], row['COD_PROG_PROFESIONAL_y'], row['COD_PROG_PROFESIONAL']] if pd.notna(x)), np.nan)
    return np.nan

df_a['COD_PROG_FINAL'] = df_a.apply(obtener_codigo, axis=1)

In [12]:
df_a = df_a[['NUM_IDENTIFICACION', 'CANT_MATERIAS', 'CANT_MAT_APRO',
       'CANT_MAT_HOMOLOGADAS', 'CICLO', 'COD_UNIDAD', 'PENSUM_L', 'Marca','COD_PROG_FINAL']]

In [13]:
df_a[df_a['NUM_IDENTIFICACION']=='1036935311']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L,Marca,COD_PROG_FINAL
1089593,1036935311,0.0,0.0,NaN,PROFESIONAL,CE267,CE267,Normal,NaN
1089594,1036935311,0.0,0.0,NaN,TECNICO,CE267,CE267,Normal,NaN
1089595,1036935311,0.0,0.0,NaN,TECNOLOGO,CE267,CE267,Normal,NaN
1089596,1036935311,23.0,23.0,23.0,TECNOLOGO,VP15G,VG002,Reconocimiento,VG15G
1089597,1036935311,20.0,20.0,1.0,PROFESIONAL,VP15G,VG002,Homologante,VP15G
1089598,1036935311,29.0,29.0,29.0,TECNICO,VP15G,VG002,Reconocimiento,VT15G


In [14]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT DISTINCT Identificacion,ciclo_sugerido,Cod_unidad_sug,pensum_limpio,Estado_Alumno 
            FROM DESERCION.Python_CUN_Simulado_cambios_ciclo
            where Estado_Alumno = 'Traslado'
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_rt = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [15]:
df_a.shape

(2552817, 9)

In [16]:
df_b = pd.merge(df_a,df_rt,how='left',left_on=['NUM_IDENTIFICACION','COD_PROG_FINAL','PENSUM_L'],
                right_on=['Identificacion','Cod_unidad_sug','pensum_limpio'])

In [17]:
df_b = df_b.drop_duplicates()

In [18]:
# Identificar los estudiantes que tienen 'Traslado' en algún ciclo, agrupando por NUM_IDENTIFICACION y CICLO
traslados = df_b[df_b['Estado_Alumno'] == 'Traslado'][['NUM_IDENTIFICACION', 'CICLO']].drop_duplicates()

# Crear una máscara con todas las combinaciones de NUM_IDENTIFICACION y CICLO que deben ser Traslado
mask = df_b.set_index(['NUM_IDENTIFICACION', 'CICLO']).index.isin(traslados.set_index(['NUM_IDENTIFICACION', 'CICLO']).index)

# Actualizar directamente todas las filas que cumplen la condición
df_b.loc[mask, 'Estado_Alumno'] = 'Traslado'


In [19]:
# df_b.loc[(df_b['Marca'].isin(['Reconocimiento','Homologante']))&
#          (df_b['Estado_Alumno']=='Traslado'),
#          'Marca'] = df_b['Estado_Alumno']

In [20]:
df_b[df_b['NUM_IDENTIFICACION']=='1036935311']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L,Marca,COD_PROG_FINAL,Identificacion,ciclo_sugerido,Cod_unidad_sug,pensum_limpio,Estado_Alumno
1089777,1036935311,0.0,0.0,NaN,PROFESIONAL,CE267,CE267,Normal,NaN,NaN,NaN,NaN,NaN,NaN
1089778,1036935311,0.0,0.0,NaN,TECNICO,CE267,CE267,Normal,NaN,NaN,NaN,NaN,NaN,NaN
1089779,1036935311,0.0,0.0,NaN,TECNOLOGO,CE267,CE267,Normal,NaN,NaN,NaN,NaN,NaN,NaN
1089780,1036935311,23.0,23.0,23.0,TECNOLOGO,VP15G,VG002,Reconocimiento,VG15G,NaN,NaN,NaN,NaN,NaN
1089781,1036935311,20.0,20.0,1.0,PROFESIONAL,VP15G,VG002,Homologante,VP15G,NaN,NaN,NaN,NaN,NaN
1089782,1036935311,29.0,29.0,29.0,TECNICO,VP15G,VG002,Reconocimiento,VT15G,NaN,NaN,NaN,NaN,NaN


In [21]:
df_final = df_b[['NUM_IDENTIFICACION', 'CANT_MATERIAS', 'CANT_MAT_APRO',
       'CANT_MAT_HOMOLOGADAS', 'CICLO','COD_PROG_FINAL','PENSUM_L',
       'Marca', 'Estado_Alumno']]

In [22]:
df_final['Estado_Alumno'].unique()
df_final[df_final['Estado_Alumno']=='Traslado']
df_final[df_final['NUM_IDENTIFICACION']=='1036935311']
# df_final['Marca'].unique()

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_PROG_FINAL,PENSUM_L,Marca,Estado_Alumno
1089777,1036935311,0.0,0.0,NaN,PROFESIONAL,NaN,CE267,Normal,NaN
1089778,1036935311,0.0,0.0,NaN,TECNICO,NaN,CE267,Normal,NaN
1089779,1036935311,0.0,0.0,NaN,TECNOLOGO,NaN,CE267,Normal,NaN
1089780,1036935311,23.0,23.0,23.0,TECNOLOGO,VG15G,VG002,Reconocimiento,NaN
1089781,1036935311,20.0,20.0,1.0,PROFESIONAL,VP15G,VG002,Homologante,NaN
1089782,1036935311,29.0,29.0,29.0,TECNICO,VT15G,VG002,Reconocimiento,NaN


In [23]:
df_final2 = df_final

In [24]:
# Asignar una prioridad numérica para facilitar el ordenamiento
prioridad = {'Normal': 1, 'Homologante': 2, 'Reconocimiento': 3}
df_final2['prioridad'] = df_final2['Marca'].map(prioridad)

# Ordenar el DataFrame por cedula, ciclo, cod_unidad, pensum y prioridad
df_final2 = df_final2.sort_values(['NUM_IDENTIFICACION', 'CICLO', 'COD_PROG_FINAL', 'PENSUM_L', 'prioridad'])

# Eliminar duplicados dejando el primer registro (que será el de mayor prioridad)
df_final2 = df_final2.drop_duplicates(subset=['NUM_IDENTIFICACION', 'CICLO', 'COD_PROG_FINAL', 'PENSUM_L'], keep='first')

# Eliminar la columna de prioridad ya que solo fue utilizada para ordenar
df_final2 = df_final2.drop(columns=['prioridad'])

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_7164\4216554024.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final2['prioridad'] = df_final2['Marca'].map(prioridad)


In [25]:
df_final2.shape

(1949742, 9)

In [26]:
df_final2[df_final2['NUM_IDENTIFICACION']=='1090398823']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_PROG_FINAL,PENSUM_L,Marca,Estado_Alumno
1181298,1090398823,30.0,22.0,0.0,PROFESIONAL,PCU12,1035,Normal,NaN
1181300,1090398823,31.0,31.0,25.0,PROFESIONAL,PVC06,1037,Homologante,NaN
1181288,1090398823,0.0,0.0,NaN,PROFESIONAL,NaN,CE277,Normal,NaN
1181299,1090398823,26.0,26.0,0.0,TECNICO,TCU12,1035,Normal,NaN
1181296,1090398823,24.0,24.0,24.0,TECNICO,VTE01,1037,Reconocimiento,NaN
1181289,1090398823,0.0,0.0,NaN,TECNICO,NaN,CE277,Normal,NaN
1181297,1090398823,19.0,19.0,0.0,TECNOLOGO,GCU12,1035,Normal,NaN
1181295,1090398823,15.0,15.0,15.0,TECNOLOGO,VGE01,1037,Reconocimiento,NaN
1181290,1090398823,0.0,0.0,NaN,TECNOLOGO,NaN,CE277,Normal,NaN


In [27]:
df_final2.loc[(df_final2['Marca']=='Reconocimiento')&(df_final2['Estado_Alumno']=='Traslado'),'Marca'] = 'Normal'

In [28]:
df_final2['CICLO'].unique()

array(['PROFESIONAL', 'TECNICO', 'TECNOLOGO'], dtype=object)

In [29]:
# No es necesario transformar a minúsculas, sólo limpiar espacios
df_final2['Marca'] = df_final2['Marca'].str.strip()
df_final2['CICLO'] = df_final2['CICLO'].str.strip()

# Diccionario para guardar clasificación final por grupo
clasificacion_dict = {}

# Agrupar por NUM_IDENTIFICACION y PENSUM_L (solo dos)
for (ident, pensum), grupo in df_final2.groupby(['NUM_IDENTIFICACION', 'PENSUM_L']):
    
    marcas = grupo.set_index('CICLO')['Marca'].to_dict()

    tiene_profesional = 'PROFESIONAL' in marcas
    tecnico_es_reconocimiento = marcas.get('TECNICO') == 'Reconocimiento'
    tecnologo_es_reconocimiento = marcas.get('TECNOLOGO') == 'Reconocimiento'
    tecnologo_no_es_reconocimiento = 'TECNOLOGO' in marcas and marcas['TECNOLOGO'] != 'Reconocimiento'
    tiene_homologante = 'Homologante' in marcas.values()

    # Aplicar prioridad
    if tiene_profesional and tecnico_es_reconocimiento and tecnologo_es_reconocimiento:
        clasificacion = 'Reconocimiento de título'
    elif tecnologo_no_es_reconocimiento and tecnico_es_reconocimiento:
        clasificacion = 'Reconocimiento de técnico'
    elif tiene_homologante:
        clasificacion = 'Homologante'
    else:
        clasificacion = 'Normal'
    
    # Usar la clave correcta del grupo
    clasificacion_dict[(ident, pensum)] = clasificacion

# Aplicar al DataFrame destino
df_final2['Clasificacion_Final'] = df_final2.apply(
    lambda row: clasificacion_dict.get((row['NUM_IDENTIFICACION'], row['PENSUM_L'])),
    axis=1
)


In [30]:
df_final2[df_final2['NUM_IDENTIFICACION']=='1110560322'].sort_values('PENSUM_L')

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_PROG_FINAL,PENSUM_L,Marca,Estado_Alumno,Clasificacion_Final
1786382,1110560322,20.0,20.0,0.0,PROFESIONAL,VP15G,VG002,Normal,NaN,Reconocimiento de título
1786383,1110560322,29.0,29.0,29.0,TECNICO,VT15G,VG002,Reconocimiento,NaN,Reconocimiento de título
1786381,1110560322,23.0,23.0,23.0,TECNOLOGO,VG15G,VG002,Reconocimiento,NaN,Reconocimiento de título


In [31]:
# df_final2.to_csv('Reconocimiento.csv',index=False)

In [32]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'RECONOCIMIENTOS'
schema_name = 'SRUBIO'

# Send the data to sql server
df_final2.to_sql(
    name=table_name,
    schema=schema_name, 
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()